In [1]:
import ttnn

2026-02-05 16:17:03.526 | DEBUG    | ttnn:<module>:77 - Initial ttnn.CONFIG:
Config{cache_path=/root/.cache/ttnn,model_cache_path=/root/.cache/ttnn/models,tmp_dir=/tmp/ttnn,enable_model_cache=false,enable_fast_runtime_mode=true,throw_exception_on_fallback=false,enable_logging=false,enable_graph_report=false,enable_detailed_buffer_report=false,enable_detailed_tensor_report=false,enable_comparison_mode=false,comparison_mode_should_raise_exception=false,comparison_mode_pcc=0.9999,root_report_path=generated/ttnn/reports,report_name=std::nullopt,std::nullopt}


In [2]:
device = ttnn.open_device(device_id=0)

2026-02-05 16:17:04.269 | info     |             UMD | Starting topology discovery. (topology_discovery.cpp:69)
2026-02-05 16:17:04.660 | info     |             UMD | Established firmware bundle version: 19.1.0 (topology_discovery.cpp:369)
2026-02-05 16:17:04.660 | info     |             UMD | Established ETH FW version: 1.7.0 (topology_discovery_blackhole.cpp:305)
2026-02-05 16:17:04.660 | info     |             UMD | Completed topology discovery. (topology_discovery.cpp:73)
2026-02-05 16:17:04.879 | info     |          Device | Opening user mode device driver (tt_cluster.cpp:209)
2026-02-05 16:17:04.880 | info     |             UMD | Starting topology discovery. (topology_discovery.cpp:69)
2026-02-05 16:17:05.274 | info     |             UMD | Established firmware bundle version: 19.1.0 (topology_discovery.cpp:369)
2026-02-05 16:17:05.274 | info     |             UMD | Established ETH FW version: 1.7.0 (topology_discovery_blackhole.cpp:305)
2026-02-05 16:17:05.274 | info     |       

In [3]:
import ttnn
import torch

# ----------------------------
# Config
# ----------------------------

file_b = "k_before_fill.tensorbin"   # path to first tensor
file_a = "keys_after_fill.tensorbin"   # path to second tensor

# Slice params: [B, H, S, D]
SLICE_SEQ = 35

RTOL = 1e-3
ATOL = 1e-3


# ----------------------------
# Load tensors
# ----------------------------
a = ttnn.load_tensor(file_a, device=device)
b = ttnn.load_tensor(file_b, device=device)

print("Loaded:")
print("A:", a.shape, a.dtype, a.layout)
print("B:", b.shape, b.dtype, b.layout)


# ----------------------------
# Materialize (important for tensorbin)
# ----------------------------
a = ttnn.clone(a)
b = ttnn.clone(b)


# ----------------------------
# Convert to torch
# ----------------------------
a_t = ttnn.to_torch(a)
b_t = ttnn.to_torch(b)

print("Torch shapes:")
print("A:", a_t.shape)
print("B:", b_t.shape)


# ----------------------------
# Slice A: [:, :, :35, :]
# ----------------------------
a_slice = a_t[:, :, :SLICE_SEQ, :]


# ----------------------------
# Match B shape if needed
# ----------------------------
if a_slice.shape != b_t.shape:
    print("Shape mismatch!")
    print("A slice:", a_slice.shape)
    print("B:", b_t.shape)
    raise RuntimeError("Shapes do not match")


# ----------------------------
# Compare
# ----------------------------
equal = torch.equal(a_slice, b_t)
close = torch.allclose(a_slice, b_t, rtol=RTOL, atol=ATOL)

print("\nComparison:")
print("Exact equal:", equal)
print("Allclose   :", close)


# ----------------------------
# Diagnostics if mismatch
# ----------------------------
if not close:
    diff = torch.abs(a_slice - b_t)

    print("\nMismatch stats:")


Loaded:
A: Shape([1, 32, 1024, 64]) DataType.BFLOAT16 Layout.TILE
B: Shape([1, 32, 35, 64]) DataType.BFLOAT16 Layout.TILE
Torch shapes:
A: torch.Size([1, 32, 1024, 64])
B: torch.Size([1, 32, 35, 64])

Comparison:
Exact equal: True
Allclose   : True


In [4]:
a_slice[0]

tensor([[[-1.8262e-01, -1.3867e-01,  3.6719e-01,  ...,  3.3984e-01,
          -6.1035e-02,  5.8594e-02],
         [-7.7148e-02, -7.6660e-02,  2.9688e-01,  ...,  2.6953e-01,
          -1.6406e-01,  8.5449e-03],
         [-1.9453e+00, -8.7402e-02, -2.8516e-01,  ..., -1.4258e-01,
          -1.0078e+00,  6.9141e-01],
         ...,
         [-1.8438e+00, -2.1484e-01, -4.1211e-01,  ...,  6.6797e-01,
           3.7891e-01, -5.1953e-01],
         [-1.2578e+00, -7.3828e-01,  1.2109e-01,  ...,  1.4766e+00,
          -2.0000e+00, -3.1836e-01],
         [ 1.5625e+00, -2.0312e-01, -1.1172e+00,  ...,  1.3359e+00,
          -3.7891e-01,  2.7148e-01]],

        [[-7.5684e-02, -1.4038e-02,  9.1797e-02,  ..., -2.4414e-02,
          -1.2402e-01,  9.0332e-02],
         [-9.8145e-02,  1.2500e-01, -7.0190e-03,  ..., -1.3672e-02,
          -2.3438e-02,  1.4355e-01],
         [ 1.8047e+00, -7.1094e-01,  8.0859e-01,  ...,  2.5469e+00,
           1.7969e+00, -6.3281e-01],
         ...,
         [ 2.5625e+00, -1

In [16]:
import ttnn
import torch

# ----------------------------
# Config
# ----------------------------

file_b = "1k_before_fill.tensorbin"   # path to first tensor
file_a = "1keys_after_fill.tensorbin"   # path to second tensor

# Slice params: [B, H, S, D]
SLICE_SEQ = 35

RTOL = 1e-3
ATOL = 1e-3


# ----------------------------
# Load tensors
# ----------------------------
a = ttnn.load_tensor(file_a, device=device)
b = ttnn.load_tensor(file_b, device=device)

print("Loaded:")
print("A:", a.shape, a.dtype, a.layout)
print("B:", b.shape, b.dtype, b.layout)


# ----------------------------
# Materialize (important for tensorbin)
# ----------------------------
# a = ttnn.clone(a)
# b = ttnn.clone(b)


# ----------------------------
# Convert to torch
# ----------------------------
a_t = ttnn.to_torch(a)
b_t = ttnn.to_torch(b)
b_t_og = b_t

b_t = torch.permute(b_t, (0,2,1,3))


print("Torch shapes:")
print("A:", a_t.shape)
print("B:", b_t.shape)


# ----------------------------
# Slice A: [:, :, :35, :]
# ----------------------------
a_slice = a_t[:, :, 44:45, :]


# ----------------------------
# Match B shape if needed
# ----------------------------
if a_slice.shape != b_t.shape:
    print("Shape mismatch!")
    print("A slice:", a_slice.shape)
    print("B:", b_t.shape)
    raise RuntimeError("Shapes do not match")


# ----------------------------
# Compare
# ----------------------------
equal = torch.equal(a_slice, b_t)
close = torch.allclose(a_slice, b_t, rtol=RTOL, atol=ATOL)

print("\nComparison:")
print("Exact equal:", equal)
print("Allclose   :", close)


# ----------------------------
# Diagnostics if mismatch
# ----------------------------
if not close:
    diff = torch.abs(a_slice - b_t)

    print("\nMismatch stats:")


Loaded:
A: Shape([1, 32, 1024, 64]) DataType.BFLOAT16 Layout.TILE
B: Shape([1, 1, 32, 64]) DataType.BFLOAT16 Layout.TILE
Torch shapes:
A: torch.Size([1, 32, 1024, 64])
B: torch.Size([1, 32, 1, 64])

Comparison:
Exact equal: True
Allclose   : True


In [17]:
a_slice.shape

torch.Size([1, 32, 1, 64])

In [18]:
b_t_og.shape

torch.Size([1, 1, 32, 64])

In [19]:
c = ttnn.load_tensor('1keys_before_fill.tensorbin', device = device)
c = ttnn.to_torch(c)



In [20]:
c.shape

torch.Size([1, 32, 1024, 64])

In [21]:
b_t_og[:, :, :, :]

tensor([[[[-0.0332, -0.0206, -0.1582,  ...,  0.2471,  0.1738,  0.0703],
          [-0.0947, -0.0488,  0.2793,  ...,  0.0603, -0.2383,  0.0981],
          [ 0.2734,  0.1445, -0.2080,  ..., -0.0535,  0.0991,  0.0562],
          ...,
          [ 0.4941, -0.4199,  0.4844,  ..., -0.1670,  0.0075,  0.8906],
          [ 0.1084,  0.5117, -0.1021,  ...,  0.1016, -0.0449,  0.0205],
          [ 0.5273,  0.7461, -0.1367,  ..., -0.3750,  0.0767,  0.8633]]]],
       dtype=torch.bfloat16)

In [22]:
a_t[:, :, 44]

tensor([[[-0.0332, -0.0206, -0.1582,  ...,  0.2471,  0.1738,  0.0703],
         [-0.0947, -0.0488,  0.2793,  ...,  0.0603, -0.2383,  0.0981],
         [ 0.2734,  0.1445, -0.2080,  ..., -0.0535,  0.0991,  0.0562],
         ...,
         [ 0.4941, -0.4199,  0.4844,  ..., -0.1670,  0.0075,  0.8906],
         [ 0.1084,  0.5117, -0.1021,  ...,  0.1016, -0.0449,  0.0205],
         [ 0.5273,  0.7461, -0.1367,  ..., -0.3750,  0.0767,  0.8633]]],
       dtype=torch.bfloat16)

In [25]:
c.shape

torch.Size([1, 32, 1024, 64])

In [26]:
c[:, :, 44]

tensor([[[-2.7188, -0.9062, -1.4688,  ...,  1.2812,  0.1758,  1.4766],
         [ 3.2188, -1.4375, -1.6484,  ...,  0.1572,  0.5195, -0.5625],
         [-0.0371,  1.5391, -0.4805,  ...,  0.7188,  1.0156, -0.3223],
         ...,
         [ 2.5469,  0.0903, -0.4570,  ...,  2.1406, -0.8008,  0.7031],
         [-0.7266,  1.3438,  0.3926,  ...,  0.3887,  0.4023, -0.6953],
         [ 1.8750,  0.2480,  0.5117,  ..., -0.0967, -0.6680, -1.0078]]],
       dtype=torch.bfloat16)